In [1]:
import os
import cv2
import torch
import joblib
import numpy as np
import pandas as pd

from PIL import Image
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models


In [2]:
class_names = [
    "Healthy",
    "Early Deficiency",
    "Critical Deficiency"
]

In [3]:
features = [
    "N", "P", "K",
    "ph",
    "soil_moisture",
    "temperature",
    "humidity",
    "rainfall",
    "sunlight_exposure"
]

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
data_df = pd.read_csv("train_data.csv")

In [6]:
class ImageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = models.efficientnet_b3(weights=None)

        self.model.classifier = nn.Sequential(
            nn.Linear(1536, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.model(x)

In [7]:
image_model = ImageModel()
image_model.load_state_dict(torch.load("image_model.pth", map_location=device))
image_model.to(device)
image_model.eval()

ImageModel(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
              (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (scale_ac

In [9]:
scaler = joblib.load("rf_scaler.pkl")
rf_model = joblib.load("rf_metadata_model.pkl")

rf_classes = rf_model.classes_ 

In [10]:
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
def image_prediction(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    pil_img = Image.fromarray(img)
    img_tensor = transform(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = image_model(img_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    return probs

In [12]:
def metadata_prediction(metadata_dict):
    df = pd.DataFrame([metadata_dict])
    scaled = scaler.transform(df[features])

    probs = rf_model.predict_proba(scaled)[0]

    # reorder to match class_names
    ordered_probs = np.zeros(len(class_names))

    for i, cls in enumerate(rf_classes):
        idx = class_names.index(cls)
        ordered_probs[idx] = probs[i]

    return ordered_probs

In [13]:
def dynamic_fusion(image_prob, metadata_prob):

    # Best stable weights
    image_weight = 0.7
    metadata_weight = 0.3

    final_prob = (
        image_weight * image_prob +
        metadata_weight * metadata_prob
    )

    return final_prob

In [14]:
def get_best_recommendation(metadata_dict, predicted_class):

    class_data = data_df[data_df["Label"] == predicted_class]

    X = class_data[features]
    y = class_data["Recommendation"].values

    # SCALE (important)
    X_scaled = scaler.transform(X)

    input_df = pd.DataFrame([metadata_dict])[features]
    input_scaled = scaler.transform(input_df)

    distances = np.linalg.norm(X_scaled - input_scaled, axis=1)

    # pick from top 5 (avoid repetition)
    top_k = np.argsort(distances)[:5]
    idx = np.random.choice(top_k)

    return y[idx]

In [15]:
def multimodal_predict(image_path, metadata_dict):

    img_prob = image_prediction(image_path)
    meta_prob = metadata_prediction(metadata_dict)

    final_prob = dynamic_fusion(img_prob, meta_prob)

    final_class = class_names[np.argmax(final_prob)]
    confidence = np.max(final_prob) * 100

    recommendation = get_best_recommendation(metadata_dict, final_class)

    # ================================
    # OUTPUT
    # ================================
    print("\n===== Crop Health Diagnosis =====")

    print("\nImage Probabilities:", np.round(img_prob, 4))
    print("Metadata Probabilities:", np.round(meta_prob, 4))
    print("Fusion Probabilities:", np.round(final_prob, 4))

    print("\nPredicted Class:", final_class)
    print(f"Confidence Score: {confidence:.2f}%")

    print("\nRecommendation:")
    print(recommendation)

In [71]:
image_path = "test/Early Deficiency/images/Early-Deficiency-58-_90_jpg.rf.d4febfd2040ae5217b7c41ef51ee2484.jpg"
metadata ={
"N": 52,
"P": 44,
"K": 43,
"ph": 5.0,
"soil_moisture": 14.3,
"temperature": 20.1,
"humidity": 36.2,
"rainfall": 45.0,
"sunlight_exposure": 7.2
}
multimodal_predict(image_path, metadata)


===== Crop Health Diagnosis =====

Image Probabilities: [0. 1. 0.]
Metadata Probabilities: [0.0131 0.5707 0.4161]
Fusion Probabilities: [0.0039 0.8712 0.1248]

Predicted Class: Early Deficiency
Confidence Score: 87.12%

Recommendation:
Apply potash fertilizer to strengthen plant resistance and improve potassium levels; Use potassium sulfate or wood ash as a potassium supplement; Increase irrigation frequency to improve soil moisture availability; Implement drip irrigation or mulching to retain soil moisture; Apply agricultural lime to increase soil pH and improve nutrient availability; Use mulching and proper irrigation practices to maintain field humidity; Maintain balanced fertilization using NPK fertilizers; Regularly monitor soil nutrients and moisture levels; Adopt proper crop rotation to maintain soil fertility; Use organic compost to enhance soil structure; Conduct periodic soil testing for optimal nutrient management
